# Baseline Poisson Model\n\nTrainiert das Team-Stärke-Poisson-Modell (Angriff/Abwehr pro Team + Heimvorteil) auf den Saisons 21/22 bis 24/25 und prüft, ob das Training konvergiert und die Team-Ratings plausibel aussehen.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.csv_source import CSVSource
from src.data.loader import DataLoader
from src.models.features import build_design_matrix
from src.models.poisson_regressor import PoissonRegressor

In [ ]:
DATA_DIR = Path.cwd().parent / "data" / "raw"
TRAIN_SEASONS = ["season-2122.csv", "season-2223.csv", "season-2324.csv", "season-2425.csv"]

frames = []
for filename in TRAIN_SEASONS:
    loader = DataLoader(CSVSource(DATA_DIR / filename))
    frames.append(loader.load())

train_df = pd.concat(frames, ignore_index=True)
train_df.shape

In [ ]:
X, y, team_index = build_design_matrix(train_df)
X.shape, y.shape, len(team_index)

In [ ]:
model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
model.fit(X, y)

In [ ]:
plt.plot(model.loss_history_)
plt.xlabel("Iteration")
plt.ylabel("Loss (mean negative log-likelihood)")
plt.title("Training loss")
plt.show()

In [ ]:
n_teams = len(team_index)
attack = model.coef_[1 : 1 + n_teams]
defense = model.coef_[1 + n_teams : 1 + 2 * n_teams]
home_advantage = model.coef_[0]

ratings = pd.DataFrame({
    "team": list(team_index.keys()),
    "attack": attack,
    "defense": defense,
}).sort_values("attack", ascending=False)

print(f"Home advantage (log-scale): {home_advantage:.3f}")
ratings

## Vergleich mit Saison 25/26\n\nSpieltag für Spieltag vorhersagen (9 Spiele pro Spieltag, nach Datum sortiert) und mit den echten Ergebnissen abgleichen. Punkte nach Kicktipp-Regeln: 4 = exaktes Ergebnis, 3 = richtige Tordifferenz (deckt auch richtig getippte Unentschieden ab), 1 = nur Tendenz richtig, 0 = daneben. `Hamburg` (Aufsteiger, keine Trainingshistorie) wird als durchschnittliches Team behandelt.

In [ ]:
from src.models.predict import predict_score
from src.models.scoring import kicktipp_points

test_df = DataLoader(CSVSource(DATA_DIR / "season-2526.csv")).load()
test_df = test_df.sort_values("date", kind="stable").reset_index(drop=True)
test_df["matchday"] = test_df.index // 9 + 1
test_df.shape

In [ ]:
results = []
for match in test_df.itertuples():
    predicted = predict_score(model, match.hometeam, match.awayteam, team_index)
    actual = (int(match.fthg), int(match.ftag))
    results.append({
        "matchday": match.matchday,
        "home": match.hometeam,
        "away": match.awayteam,
        "predicted": f"{predicted[0]}:{predicted[1]}",
        "actual": f"{actual[0]}:{actual[1]}",
        "points": kicktipp_points(predicted, actual),
    })

results_df = pd.DataFrame(results)
results_df.head(9)

In [ ]:
points_per_matchday = results_df.groupby("matchday")["points"].sum()

plt.plot(points_per_matchday.index, points_per_matchday.values, marker="o")
plt.xlabel("Spieltag")
plt.ylabel("Punkte")
plt.title("Kicktipp-Punkte pro Spieltag (Saison 25/26)")
plt.show()

print(f"Gesamtpunkte: {results_df['points'].sum()} ({results_df['points'].mean():.2f} im Schnitt pro Spiel)")
print("Verteilung:")
print(results_df["points"].value_counts().sort_index(ascending=False))